# Physics-Informed Neural Networks

In [ ]:
import notebook as nb
print(nb.__version__)

In [ ]:
import types
def imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            yield val.__name__ 
list(imports())

We consider in the following the use of Physics-Informed Neural Networks(PINNs) for the construction of a surrogate model for the solution to the linear advection equation in one space dimension.
$u_{t}+cu_{x}=0 \qquad  t ∈ \mathbb{R^+}, x ∈ \mathbb{R}$ 
This problem have analytical solutions in the form $u(x,t)=f(x-ct)$, and an initial condition of the form $u(x,0)=f(x)=\sin(\pi x)$ is assumed. For $c>0$, a boundary condition is imposed on the left boundary because the information is travelling from left to right in this case
$u(0,t)=f(-ct)=\sin(-\pi ct)$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time 
# High precision
torch.set_default_dtype(torch.float64)

# Define device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA on GPU")
# Check for MPS availability next if CUDA is not available (for macOS with Apple Silicon)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS on Apple Silicon GPU")
    torch.set_default_dtype(torch.float32) # mps does not allow for float64
# Default to CPU if neither CUDA nor MPS is available
else:
    device = torch.device("cpu")
    print("Using CPU")

# Define the neural network model
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 30),
            nn.Tanh(),
            nn.Linear(30, 30),
            nn.Tanh(),
            nn.Linear(30, 30),
            nn.Tanh(),
            nn.Linear(30, 30),
            nn.Tanh(),
            nn.Linear(30, 30),
            nn.Tanh(),
            nn.Linear(30, 1)
        )

    def forward(self, x, t):
        xt = torch.cat((x, t), dim=1)
        u = self.net(xt)
        return u

# Helper functions to calculate derivatives
def grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True, retain_graph=True)[0]

# Physics-informed loss
def loss_fn(model, x, t, eps):
    x.requires_grad_(True)
    t.requires_grad_(True)

    u = model(x, t)

    u_t = grad(u, t)
    u_x = grad(u, x)
    u_xx = grad(u_x, x)

    # PDE loss
    f = u_t + u * u_x - eps * u_xx  
    loss_pde = torch.mean(f**2)

    # Initical condition
    initial_condition = model(x, torch.zeros_like(t)) - (-torch.tanh((x+0.5)/(2*eps)) + 1)
    loss_initial = torch.mean(initial_condition**2)

    # Boundary conditions
    boundary_condition_left = model(-torch.ones_like(x), t) - (-torch.tanh((-0.5-t)/(2*eps)) + 1)
    boundary_condition_right = model(torch.ones_like(x), t) - (-torch.tanh((1.5-t)/(2*eps)) + 1)
    loss_boundary = torch.mean(boundary_condition_left**2) + torch.mean(boundary_condition_right**2)

    return loss_pde + 10.0*loss_boundary + 10.0*loss_initial
    
# Training
def train(model, epochs, optimizer, x, t,c):
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        l = loss_fn(model, x, t,c)
        l.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f'Epoch {epoch}: Loss = {l.item()}')

# Grid points
xmin, xmax, nx = -1, 1, 200
tmin, tmax, nt = 0, 10, 200
x = torch.linspace(xmin, xmax, nx, device=device).unsqueeze(1)
t = torch.linspace(tmin, tmax, nt, device=device).unsqueeze(1)
x_mesh, t_mesh = torch.meshgrid(x.flatten(), t.flatten())
x_mesh, t_mesh = x_mesh.flatten().unsqueeze(1), t_mesh.flatten().unsqueeze(1)

# Constants
eps = 0.01/np.pi

# Initialize model, optimizer
model = PINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

t0 = time.time()
# Train the model
train(model, 15000, optimizer, x_mesh, t_mesh, eps)

t1 = time.time()
print(f"training time: {t1-t0}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# Assuming 'model' and 'device' are already defined and the model is trained.

# Generate predictions from the trained model
model.eval()  # Set the model to evaluation mode

# Exact solution (update this according to your problem's exact solution)
u_exact = -np.tanh(
    (x_mesh.cpu().detach().numpy()+0.5-t_mesh.cpu().detach().numpy())
    /
    (2*eps)
    )+1

x_pt = torch.tensor([[0.0]], device = device, requires_grad=True)
t_pt = torch.tensor([[1.6307/np.pi]], device = device, requires_grad = True)
u_pt = model(x_pt, t_pt)
du_pt = np.float64(torch.autograd.grad(u_pt, x_pt)[0])

#NN gradient
x_mesh.requires_grad_(True)
u_pred = model(x_mesh, t_mesh)
du_pred = (torch.autograd.grad(u_pred, x_mesh, grad_outputs=torch.ones_like(x_mesh))[0]).cpu().detach().numpy()
u_pred = u_pred.cpu().detach().numpy()
#analytical gradient
x_mesh_np = x_mesh.cpu().detach().numpy()
t_mesh_np = t_mesh.cpu().detach().numpy()
du_exact = -1/(2*eps) * (1/np.cosh((x_mesh_np + 0.5 - t_mesh_np)/(2*eps)))**2


# Compute L2 errors
l2_err = np.sum((u_exact - u_pred)**2)
l2_err_t0 = np.sum((u_exact[0, :] - u_pred[0, :])**2)
l2_err_tmax = np.sum((u_exact[-1, :] - u_pred[-1, :])**2)



# Output error information
print(f'L2 err, all timesteps: {l2_err}')
print(f'Avg-L2 err, all timesteps: {l2_err/u_pred.size}')
print(f'L2 err, t=0: {l2_err_t0}')
print(f'L2 err, t={tmax}: {l2_err_tmax}')
print(f"u_x at x=0 t=1.6307/pi: {du_pt}, should be -152.00516 ")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming your x_mesh and t_mesh are defined correctly in the tensor form.
x = torch.linspace(xmin, xmax, nx, device=device).cpu().numpy()
t = torch.linspace(tmin, tmax, nt, device=device).cpu().numpy()

# Since these are grid centers, calculate the step size and expand the grid to cover edges.
x_step = (x[1] - x[0])
t_step = (t[1] - t[0])

# Create new arrays that represent the edges, not the centers
x_edges = np.append(x, x[-1] + x_step) - x_step/2
t_edges = np.append(t, t[-1] + t_step) - t_step/2

# Use these arrays for pcolormesh
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 15))

# Exact solution
ax = axes[0][0]
cc = ax.pcolormesh(x_edges, t_edges, u_exact.reshape((nt, nx)), cmap='seismic', shading='auto')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('$u(x,t) (exact)$')
fig.colorbar(cc, ax=ax)

# Predicted solution
ax = axes[0][1]
cc = ax.pcolormesh(x_edges, t_edges, u_pred.reshape((nt, nx)), cmap='seismic', shading='auto')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('$NN(x,t) \simeq u(x,t) (prediction)$')
fig.colorbar(cc, ax=ax)

# Errors
ax = axes[1][0]
cc = ax.pcolormesh(x_edges, t_edges, u_exact.reshape((nt, nx)) - u_pred.reshape((nt, nx)), cmap='seismic', shading='auto')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title(f'Error w. mean err = {np.mean(np.abs(u_pred - u_exact)):.3f}')
fig.colorbar(cc, ax=ax)

# Gradients
ax = axes[1][1]
cc = ax.pcolormesh(x_edges, t_edges, du_exact.reshape((nt,nx)) - du_pred.reshape((nt,nx)), cmap='seismic', shading='auto')

flat_idx = np.argmax(du_exact.reshape((nt,nx)) - du_pred.reshape((nt,nx)))

row, col = np.unravel_index(flat_idx, (nt, nx))
ax.plot(x[col], t[row], '.', markersize=15, label='max err')

ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title(f'Gradient w. max err = {np.max(np.abs(du_exact - du_pred)):.3f}')
fig.colorbar(cc, ax=ax)

plt.savefig("img/pinn_simple.png")